# S6E10 Airline Satisfaction: final pipeline (v4)

**What is left to run, in order** (everything we already measured is in the table at the bottom and in the commented appendix):
1. **Setup**: attach the three datasets, run the script cell.
2. **A. Two checks** (~4 min): is the original-data stack's LB gap noise or a train/test difference?
3. **B. Quick mixed stack** (~2 min if `og5/` still exists, else ~21 min).
4. **C. Final 10-fold run** with the original data (~45-50 min, a guess), then the mixed final stack (~3 min).
5. **Pick two submissions** (notes in D).

**Datasets to attach:** the competition; `yukrxfo/airplane` (your saved runs); the original data (`teejmahal20/airline-passenger-satisfaction`).
Where one cell depends on files from an earlier cell or session, the cell says which.

In [ ]:
%%writefile s6e10_v4.py
"""Playground S6E10 (Airline Satisfaction), v4: new-idea lab + full pipeline (ROC AUC).

Built on the v3 code that ran on real data (features, leak-free target encoding, caching, stacker, trees).
New in v4 (every item is a HYPOTHESIS until its stage below has been run; see the notebook for the evidence):
  * tabm_*      own TabM-style net: BatchEnsemble MLP, k heads, PLE inputs + route embeddings, EMA, no
                best-epoch selection on the validation fold (removes that optimistic bias)
  * realmlp_long / tabm_pk   pytabkit models with a far longer schedule than v3 (optional, needs pip install)
  * --te2       hierarchical-shrinkage TE (fd -> fd_age -> identity, fd -> flight), neighbouring-distance
                kernel TE, per-route rating deviations
  * --knn       GPU k-nearest-neighbour label features ("sibling retrieval"), leave-one-out inside the fold
  * xgb_rank    XGBoost pairwise-ranking objective (optimises ordering, i.e. AUC, instead of log-loss)
  * --stack2    LightGBM level-2 stacker on member logits + meta features, compared fairly with the linear stacker
Stages (--stage): diag (data diagnostics), ablate (fold-1 feature ablation), bench (fold-1 NN benchmark), full.
Run with plain `python s6e10_v4.py [--stage ...]` (threads drive 2 GPUs + 4 CPU cores).
"""
import argparse
import hashlib
import inspect
import logging
import threading
import time
import traceback
import warnings
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from functools import partial
from pathlib import Path
from typing import Callable

import catboost as cb
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception:  # neural members and kNN features are skipped without a working torch
    torch = None

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
for _n in ("pytorch_lightning", "lightning", "lightning.pytorch"):
    logging.getLogger(_n).setLevel(logging.ERROR)
SEED = 42
CPU_THREADS = 4
FEATURE_VERSION = 5  # bump when feature definitions change, so stale cache files are ignored
T0 = time.time()
REF_FOLD1 = "fold-1 reference (v3 log): lgb_route 0.96057 | xgb_deep 0.96048 | cat_native 0.96023 | v3 MLPs 0.9591-0.9593"


def log(msg: str) -> None:
    print(f"[{time.time() - T0:5.0f}s] {msg}", flush=True)


# --------------------------------------------------------------------------- config
@dataclass
class Cfg:
    data: str
    out: str
    target: str
    folds: int
    rounds: int       # max boosting rounds (early stopping decides the real number)
    nn_epochs: int    # epochs used in --quick mode
    quick: bool
    only: list
    stage: str
    te2: bool
    knn: bool
    knn_preset: int
    stack2: bool
    seeds: int        # NN seeds per fold (averaged inside each neural member)
    abl_seeds: int
    imports: list = field(default_factory=list)
    nn_name: str = "tabm_reg"
    pl_grid: str = "1.0:0,1.0:0.4"
    ngpu: int = 0
    og: str = ""
    og_te: bool = False
    og_knn: bool = False


def make_cfg() -> Cfg:
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="/kaggle/input/competitions/playground-series-s6e10")
    p.add_argument("--out", default=".")
    p.add_argument("--target", default="satisfaction")
    p.add_argument("--stage", default="full", choices=["full", "diag", "ablate", "bench", "boost", "adv"])
    p.add_argument("--folds", type=int, default=5, help="10 is slightly better but about twice as slow")
    p.add_argument("--only", default="", help="comma-separated member names (bench/full)")
    p.add_argument("--te2", action="store_true", help="hierarchical + kernel TE and rating deviations")
    p.add_argument("--knn", action="store_true", help="GPU kNN label features")
    p.add_argument("--knn-preset", type=int, default=0, choices=[0, 1, 2])
    p.add_argument("--stack2", action="store_true", help="also try a LightGBM level-2 stacker")
    p.add_argument("--nn-seeds", type=int, default=1)
    p.add_argument("--abl-seeds", type=int, default=2)
    p.add_argument("--import-npz", default="", help="comma-separated oof_test_logits.npz files from earlier runs of "
                                                      "YOUR OWN models; their members join the stack")
    p.add_argument("--nn-name", default="tabm_reg", help="net used by --stage boost")
    p.add_argument("--pl-grid", default="1.0:0,1.0:0.4",
                   help="pseudo-label settings weight:min_confidence, comma-separated (--stage boost)")
    p.add_argument("--og", default="", help="original airline-satisfaction dataset: a folder with train.csv/test.csv, "
                                           "or 'auto' to search /kaggle/input (add the dataset to the notebook first)")
    p.add_argument("--og-te", action="store_true", help="also add target encodings built from the original rows")
    p.add_argument("--og-knn", action="store_true", help="also add kNN labels from the original rows (GPU)")
    p.add_argument("--quick", action="store_true", help="short smoke test: 2 folds, 100 rounds, 2 epochs "
                                                        "(models are under-trained, scores are NOT representative)")
    a = p.parse_args()
    try:
        ngpu = torch.cuda.device_count() if torch else 0
    except Exception:
        ngpu = 0
    q = a.quick
    return Cfg(a.data, a.out, a.target, 2 if q else a.folds, 100 if q else 6000, 2, q,
               [s for s in a.only.split(",") if s], a.stage, a.te2, a.knn, a.knn_preset, a.stack2,
               a.nn_seeds, 1 if q else a.abl_seeds, [s for s in a.import_npz.split(",") if s], a.nn_name, a.pl_grid, ngpu, a.og, a.og_te, a.og_knn)


def gpu(cfg: Cfg, i: int) -> str:
    return f"cuda:{min(i, cfg.ngpu - 1)}" if cfg.ngpu else "cpu"


def cat_device(cfg: Cfg, i: int) -> dict:
    if cfg.ngpu:
        return dict(task_type="GPU", devices=str(min(i, cfg.ngpu - 1)), gpu_ram_part=0.45)
    return dict(thread_count=CPU_THREADS)


# --------------------------------------------------------------------------- data + features
@dataclass
class Feats:
    base: np.ndarray     # raw columns + zero-rating count + delay recovery features
    route: np.ndarray    # route-context statistics per Flight Distance (train+test, no labels)
    keys: np.ndarray     # integer codes of the target-encoding keys, one column each
    fd_code: np.ndarray  # Flight Distance as a categorical code
    ids: np.ndarray      # id columns for NN embeddings: route and route x segment (rare values -> 0)
    vocabs: list         # vocabulary size of each id column
    cat_idx: list        # raw categorical column indices (for CatBoost / RealMLP)
    n: int               # number of train rows (test rows follow)
    key_names: list = field(default_factory=list)
    rdev: np.ndarray = None      # rating minus route-mean rating (label-free)
    parts: dict = field(default_factory=dict)  # standardised blocks for the kNN retrieval space
    meta: np.ndarray = None      # a few raw columns for the level-2 model
    fd_rank: np.ndarray = None   # rank of each Flight Distance code in sorted distance order
    use_rdev: bool = False
    enc_names: list = field(default_factory=list)
    enc_sig: str = ""
    og: np.ndarray = None        # features from the ORIGINAL dataset (label-free for the competition rows)
    og_names: list = field(default_factory=list)
    use_og: bool = False
    og_sig: str = ""


def find(cols: list, exact: str = None, has: tuple = ()):
    low = [c.lower() for c in cols]
    if exact and exact in low:
        return low.index(exact)
    for i, c in enumerate(low):
        if has and all(h in c for h in has):
            return i
    return None


def cross(*arrs) -> np.ndarray:
    """Integer code of the combination of several columns."""
    code = pd.factorize(arrs[0])[0].astype(np.int64)
    for a in arrs[1:]:
        c = pd.factorize(a)[0].astype(np.int64)
        code = pd.factorize(code * (c.max() + 1) + c)[0].astype(np.int64)
    return code


def load(cfg: Cfg):
    d = Path(cfg.data)
    train, test, sub = (pd.read_csv(d / f) for f in ("train.csv", "test.csv", "sample_submission.csv"))
    tcol = cfg.target if cfg.target in train else sub.columns[-1]
    y = train.pop(tcol).astype(str).str.lower().isin(["true", "1", "1.0", "satisfied"]).astype(np.int8).to_numpy()
    X = pd.concat([train, test.drop(columns=[c for c in (tcol,) if c in test])], ignore_index=True)
    X = X.drop(columns=[c for c in ("id", "Unnamed: 0") if c in X])
    log(f"train={len(train):,} test={len(test):,} features={X.shape[1]} positive rate={y.mean():.4f}")

    rating = []  # 0-5 survey ratings, found by their values rather than by name
    for i, c in enumerate(X.columns):
        s = X[c]
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            v = s.dropna()
            if 5 <= v.nunique() <= 6 and v.min() >= 0 and v.max() <= 5 and (v % 1 == 0).all():
                rating.append(i)
    cat_idx = [i for i, c in enumerate(X.columns) if not pd.api.types.is_numeric_dtype(X[c])
               or pd.api.types.is_bool_dtype(X[c])]
    cat_maps = {}
    for i in cat_idx:
        c = X.columns[i]
        cat_maps[c] = list(X[c].astype("category").cat.categories)
        X[c] = X[c].astype("category").cat.codes.astype(np.int16)
    X = X.fillna(X.median(numeric_only=True))
    return X, y, test, sub, tcol, rating, cat_idx, cat_maps


def rare_to_zero(code: np.ndarray, min_count: int = 5):
    """Re-index a code so values seen fewer than min_count times share id 0. Returns (ids, vocab size)."""
    cnt = np.bincount(code)
    keep = np.flatnonzero(cnt >= min_count)
    remap = np.zeros(len(cnt), np.int64)
    remap[keep] = np.arange(1, len(keep) + 1)
    return remap[code], len(keep) + 1


def zscore(v: np.ndarray) -> np.ndarray:
    v = v.astype(np.float64)
    return ((v - v.mean()) / (v.std() + 1e-9)).astype(np.float32)


def onehot(v: np.ndarray) -> np.ndarray:
    """One-hot scaled so that a category mismatch has distance 1 in the kNN retrieval space."""
    c = pd.factorize(v)[0]
    return ((c[:, None] == np.arange(c.max() + 1)).astype(np.float32)) / np.sqrt(2.0)


def build_features(X: pd.DataFrame, n: int, rating: list, cat_idx: list, cfg: Cfg) -> Feats:
    cols = list(X.columns)
    raw = X.to_numpy(np.float32)
    ix = dict(fd=find(cols, has=("distance",)), age=find(cols, exact="age"), cls=find(cols, exact="class"),
              trv=find(cols, has=("travel",)), cus=find(cols, has=("customer",)),
              gen=find(cols, exact="gender"), dep=find(cols, has=("delay", "depart")),
              arr=find(cols, has=("delay", "arriv")))
    col = lambda k: raw[:, ix[k]] if ix[k] is not None else None
    N = len(raw)

    # base: raw + proven signals (zero rating = "service not used", delay recovery)
    base = [raw]
    R = raw[:, rating] if len(rating) >= 3 else None
    if R is not None:
        base.append((R == 0).sum(1))
    dep, arr = col("dep"), col("arr")
    if dep is not None and arr is not None:
        base += [dep - arr, dep + arr, (arr + 1) / (np.abs(dep) + 1)]
    base = np.nan_to_num(np.column_stack(base).astype(np.float32))

    fd = col("fd")
    if fd is None:
        log("WARNING: no Flight Distance column found, route features disabled")
        return Feats(base, np.zeros((N, 0), np.float32), np.zeros((N, 0), np.int64), np.zeros(N, np.int64),
                     np.zeros((N, 1), np.int64), [1], cat_idx, n, [], np.zeros((N, 0), np.float32), {},
                     np.zeros((N, 1), np.float32), np.zeros(1, np.int64), False)

    # route context: per Flight Distance statistics over train+test (no labels involved)
    fd_code = pd.factorize(fd)[0].astype(np.int64)
    cnt = np.bincount(fd_code)
    total = R.sum(1) if R is not None else None
    stats = {k: v for k, v in dict(age=col("age"), dep=dep, arr=arr, rating=total, cls=col("cls"),
                                   trv=col("trv"), cus=col("cus")).items() if v is not None}
    route = [cnt[fd_code].astype(np.float32)]
    for k, v in stats.items():
        mean = (np.bincount(fd_code, weights=v) / cnt)[fd_code]
        route.append(mean)
        if k in ("age", "dep", "arr", "rating"):
            var = (np.bincount(fd_code, weights=v.astype(np.float64) ** 2) / cnt)[fd_code] - mean ** 2
            route += [np.sqrt(np.maximum(var, 0)), v - mean]

    # target-encoding keys: Flight Distance and its crosses
    parts = {k: col(k) for k in ("age", "cls", "trv", "cus", "gen")}
    keys = {"fd": fd_code}
    for k in ("cls", "trv", "cus", "age"):
        if parts[k] is not None:
            keys[f"fd_{k}"] = cross(fd, parts[k])
    if all(v is not None for v in parts.values()):
        keys["identity"] = cross(fd, *[parts[k] for k in ("age", "gen", "cls", "trv", "cus")])
    if dep is not None and arr is not None:
        keys["flight"] = cross(fd, dep, arr)
    for k in ("fd_age", "identity", "flight"):
        if k in keys:
            route.append(np.bincount(keys[k])[keys[k]].astype(np.float32))
    route = np.nan_to_num(np.column_stack(route).astype(np.float32))

    # label-free extras: rating deviations from the route mean, kNN retrieval blocks, meta columns, distance ranks
    rdev = np.zeros((N, 0), np.float32)
    if R is not None:
        rdev = np.column_stack([R[:, j] - (np.bincount(fd_code, weights=R[:, j]) / cnt)[fd_code]
                                for j in range(R.shape[1])]).astype(np.float32)
    kp = {"fd": zscore(fd)[:, None]}
    if parts["age"] is not None:
        kp["age"] = zscore(parts["age"])[:, None]
    cats = [onehot(parts[k]) for k in ("cls", "trv", "cus", "gen") if parts[k] is not None]
    if cats:
        kp["cat"] = np.hstack(cats)
    if R is not None:
        kp["rate"] = np.column_stack([zscore(R[:, j]) for j in range(R.shape[1])])
    if dep is not None and arr is not None:
        kp["delay"] = np.column_stack([zscore(np.log1p(np.maximum(dep, 0))), zscore(np.log1p(np.maximum(arr, 0)))])
    zero = np.zeros(N, np.float32)
    meta = np.column_stack([fd, cnt[fd_code], (R == 0).sum(1) if R is not None else zero,
                            parts["age"] if parts["age"] is not None else zero,
                            parts["cls"] if parts["cls"] is not None else zero]).astype(np.float32)
    uv = np.zeros(len(cnt))
    uv[fd_code] = fd
    order = np.argsort(uv, kind="stable")
    fd_rank = np.empty(len(cnt), np.int64)
    fd_rank[order] = np.arange(len(cnt))

    # NN embedding ids: route and route x segment (rare values share id 0)
    ids, vocabs = zip(*[rare_to_zero(keys[k]) for k in ("fd", "fd_cls", "fd_trv", "fd_cus") if k in keys])
    log(f"features: base={base.shape[1]} route={route.shape[1]} rdev={rdev.shape[1]} te_keys={list(keys)} "
        f"nn_id_vocabs={list(vocabs)} (ratings={len(rating)})")
    return Feats(base, route, np.column_stack(list(keys.values())), fd_code, np.column_stack(ids),
                 list(vocabs), cat_idx, n, list(keys), rdev, kp, meta, fd_rank, cfg.te2 and rdev.shape[1] > 0)



# --------------------------------------------------------------------------- original dataset (ORIGINAL airline data)
# The competition rows were generated from the 130k-row original dataset. Models fitted ONLY on those original rows
# and applied to the competition rows involve no competition label, so these features are leak-free in any CV.
def find_og(cfg: Cfg) -> Path:
    if cfg.og and cfg.og != "auto":
        return Path(cfg.og)
    comp = Path(cfg.data).resolve()
    for tr in sorted(Path("/kaggle/input").glob("**/train.csv")):
        d = tr.parent.resolve()
        if d == comp or comp in d.parents or not (d / "test.csv").exists():
            continue
        head = pd.read_csv(tr, nrows=20)
        if "satisfaction" in head.columns and head["satisfaction"].astype(str).str.contains("satisf").any():
            return d
    raise SystemExit("original dataset not found: add 'Airline Passenger Satisfaction' (train.csv/test.csv with a "
                     "'satisfaction' column of 'satisfied' / 'neutral or dissatisfied') to the notebook, or pass --og DIR")


def load_og(cfg: Cfg, X: pd.DataFrame, cat_maps: dict):
    """Original rows in the competition's column order with the competition's category codes."""
    d = find_og(cfg)
    og = pd.concat([pd.read_csv(d / f) for f in ("train.csv", "test.csv") if (d / f).exists()], ignore_index=True)
    norm = lambda t: "".join(ch for ch in str(t).lower() if ch.isalnum())
    names = {norm(c): c for c in og.columns}
    missing = [c for c in X.columns if norm(c) not in names]
    if missing or "satisfaction" not in names:
        raise SystemExit(f"original dataset lacks columns {missing or ['satisfaction']}; has {list(og.columns)}")
    cols = {}
    for c in X.columns:
        s = og[names[norm(c)]]
        if c in cat_maps:
            lut = {str(v).lower().strip(): i for i, v in enumerate(cat_maps[c])}
            codes = s.astype(str).str.lower().str.strip().map(lut)
            if codes.isna().mean() > 0.01:
                log(f"WARNING original column '{c}': {codes.isna().mean():.1%} values not in the competition's categories")
            cols[c] = codes.fillna(-1).astype(np.int16)
        else:
            cols[c] = pd.to_numeric(s, errors="coerce").fillna(X[c].median())
    Xog = pd.DataFrame(cols)
    yog = (og[names["satisfaction"]].astype(str).str.lower().str.strip() == "satisfied").astype(np.int8)
    keep = ~pd.concat([Xog, yog.rename("y")], axis=1).duplicated().to_numpy()
    log(f"original data: {len(og):,} rows from {d}, {int(keep.sum()):,} after de-duplication, "
        f"positive rate {yog[keep].mean():.4f}")
    return Xog[keep].to_numpy(np.float32), yog[keep].to_numpy()


def og_pred_features(cfg: Cfg, Xog: np.ndarray, yog: np.ndarray, Xc: np.ndarray, y: np.ndarray, n: int):
    """XGBoost + LightGBM fitted on the original rows only (5-fold average), applied to the competition rows."""
    px, pl = np.zeros(len(Xc)), np.zeros(len(Xc))
    oof_auc = []
    xp = dict(objective="binary:logistic", eval_metric="auc", eta=0.03, max_depth=8, subsample=0.8,
              colsample_bytree=0.6, min_child_weight=3, reg_lambda=2.0, tree_method="hist", device=gpu(cfg, 0),
              seed=SEED)
    lp = dict(objective="binary", learning_rate=0.03, num_leaves=63, feature_fraction=0.6, bagging_fraction=0.8,
              bagging_freq=1, lambda_l2=2.0, min_child_samples=20, num_threads=CPU_THREADS, verbose=-1, seed=SEED)
    rounds = 100 if cfg.quick else 3000
    for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xog, yog):
        da = xgb.QuantileDMatrix(Xog[a], yog[a])
        db = xgb.QuantileDMatrix(Xog[b], yog[b], ref=da)
        bx = xgb.train(xp, da, rounds, evals=[(db, "og")], early_stopping_rounds=100, verbose_eval=False)
        px += bx.predict(xgb.DMatrix(Xc), output_margin=True, iteration_range=(0, bx.best_iteration + 1)) / 5
        oof_auc.append(float(bx.best_score))
        la = lgb.Dataset(Xog[a], yog[a])
        bl = lgb.train(lp, la, rounds, valid_sets=[lgb.Dataset(Xog[b], yog[b], reference=la)],
                       callbacks=[lgb.early_stopping(100, verbose=False)])
        pl += bl.predict(Xc, raw_score=True, num_iteration=bl.best_iteration) / 5
    log(f"original-only models: AUC inside the original data {np.mean(oof_auc):.4f}; "
        f"AUC on the competition labels: xgb {roc_auc_score(y, px[:n]):.5f} lgb {roc_auc_score(y, pl[:n]):.5f}")
    return px.astype(np.float32), pl.astype(np.float32)


def og_te_features(Fe: Feats, X: pd.DataFrame, Xog: np.ndarray, yog: np.ndarray, m: float = 10.0):
    """Original-data target means per route (hierarchically shrunk): fd, fd x class, fd x travel, plus the route's count."""
    ix = col_index(list(X.columns))
    comp = {k: X.iloc[:, ix[k]].to_numpy() for k in ("fd", "cls", "trv") if ix[k] is not None}
    ogv = {k: Xog[:, ix[k]] for k in comp}
    p0 = float(yog.mean())
    cols, est = [], {}
    for name, key, parent in (("om_fd", ["fd"], None), ("om_fd_cls", ["fd", "cls"], "om_fd"),
                              ("om_fd_trv", ["fd", "trv"], "om_fd")):
        if not all(k in comp for k in key):
            continue
        g = pd.DataFrame({k: ogv[k] for k in key})
        g["y"] = yog
        agg = g.groupby(key)["y"].agg(["sum", "count"])
        j = pd.DataFrame({k: comp[k] for k in key}).join(agg, on=key)
        s, c = j["sum"].fillna(0).to_numpy(), j["count"].fillna(0).to_numpy()
        prior = est[parent] if parent else p0
        est[name] = (s + m * prior) / (c + m)
        cols.append((name, est[name]))
        if name == "om_fd":
            cols.append(("om_n_fd", np.log1p(c)))
    return [c for c, _ in cols], np.column_stack([v for _, v in cols]).astype(np.float32)


def og_knn_features(cfg: Cfg, X: pd.DataFrame, Xog: np.ndarray, yog: np.ndarray, rating: list):
    """Mean original label of the k nearest ORIGINAL rows (same retrieval space as --knn, standardised with the
    competition's statistics). Tests whether competition rows sit next to their source rows."""
    ix = col_index(list(X.columns))
    w = KNN_PRESETS[cfg.knn_preset]

    def space(vals, st):
        def z(v, key):
            if key not in st:
                st[key] = (float(np.mean(v)), float(np.std(v)) + 1e-9)
            return ((np.asarray(v, np.float64) - st[key][0]) / st[key][1]).astype(np.float32)

        def oh(v, key):
            if key not in st:
                st[key] = int(np.max(v)) + 1
            return (np.asarray(v).astype(np.int64)[:, None] == np.arange(st[key])).astype(np.float32) / np.sqrt(2.0)
        g = {}
        g["fd"] = z(vals["fd"], "fd")[:, None]
        if vals.get("age") is not None:
            g["age"] = z(vals["age"], "age")[:, None]
        cats = [oh(vals[k], k) for k in ("cls", "trv", "cus", "gen") if vals.get(k) is not None]
        if cats:
            g["cat"] = np.hstack(cats)
        if vals.get("R") is not None:
            g["rate"] = np.column_stack([z(vals["R"][:, j], f"r{j}") for j in range(vals["R"].shape[1])])
        if vals.get("dep") is not None and vals.get("arr") is not None:
            g["delay"] = np.column_stack([z(np.log1p(np.maximum(vals["dep"], 0)), "dep"),
                                          z(np.log1p(np.maximum(vals["arr"], 0)), "arr")])
        return np.hstack([g[k] * w[k] for k in w if k in g]).astype(np.float32)

    def vals_of(A):
        v = {k: A[:, ix[k]] for k in ("fd", "age", "cls", "trv", "cus", "gen", "dep", "arr") if ix[k] is not None}
        v["R"] = A[:, rating] if len(rating) >= 3 else None
        return v
    st = {}
    Zc = space(vals_of(X.to_numpy(np.float32)), st)
    Zo = space(vals_of(Xog), st)
    Z = np.vstack([Zc, Zo])
    yall = np.concatenate([np.zeros(len(Zc), np.int8), yog])
    f = knn_label_feats(Z, yall, np.arange(len(Zc), len(Z)), np.arange(len(Zc)), False, gpu(cfg, 0))
    return ["ogk_5", "ogk_20", "ogk_50", "ogk_d5", "ogk_d50"], f


def attach_og(cfg: Cfg, X: pd.DataFrame, cat_maps: dict, rating: list, Fe: Feats, y: np.ndarray, everything: bool) -> None:
    """Compute (or load from disk) the original-data features and attach them to Fe. `everything` builds all groups
    (used by --stage ablate)."""
    Xog, yog = load_og(cfg, X, cat_maps)
    want_te, want_knn = everything or cfg.og_te, (everything or cfg.og_knn) and torch is not None
    sig = repr((X.shape, Xog.shape, round(float(Xog.sum()), 1), int(yog.sum()), want_te, want_knn, cfg.knn_preset, cfg.quick))
    path = Path(cfg.out) / f"og_features_{hashlib.md5(sig.encode()).hexdigest()[:8]}.npz"
    if path.exists():
        z = np.load(path)
        Fe.og, Fe.og_names = z["og"], [str(v) for v in z["names"]]
        log(f"original-data features loaded from {path.name}: {Fe.og_names}")
    else:
        Xc = X.to_numpy(np.float32)
        px, pl = og_pred_features(cfg, Xog, yog, Xc, y, Fe.n)
        mean = (px + pl) / 2
        cnt = np.bincount(Fe.fd_code)
        rm = (np.bincount(Fe.fd_code, weights=mean) / cnt)[Fe.fd_code]
        rv = (np.bincount(Fe.fd_code, weights=mean.astype(np.float64) ** 2) / cnt)[Fe.fd_code] - rm ** 2
        cols, names = [px, pl, rm, np.sqrt(np.maximum(rv, 0)), mean - rm], \
            ["og_xgb", "og_lgb", "og_route_mean", "og_route_std", "og_dev"]
        arr = np.column_stack(cols).astype(np.float32)
        if want_te:
            n2, a2 = og_te_features(Fe, X, Xog, yog)
            names, arr = names + n2, np.hstack([arr, a2])
        if want_knn:
            n3, a3 = og_knn_features(cfg, X, Xog, yog, rating)
            names, arr = names + n3, np.hstack([arr, a3])
        Fe.og, Fe.og_names = np.nan_to_num(arr), names
        Path(cfg.out).mkdir(parents=True, exist_ok=True)
        np.savez(path, og=Fe.og, names=np.array(names))
        log(f"original-data features: {names}")
    Fe.use_og = not everything
    Fe.og_sig = hashlib.md5(Fe.og.tobytes()).hexdigest()[:8] + repr(Fe.og_names)


def col_index(cols: list) -> dict:
    return dict(fd=find(cols, has=("distance",)), age=find(cols, exact="age"), cls=find(cols, exact="class"),
                trv=find(cols, has=("travel",)), cus=find(cols, has=("customer",)), gen=find(cols, exact="gender"),
                dep=find(cols, has=("delay", "depart")), arr=find(cols, has=("delay", "arriv")))


# --------------------------------------------------------------------------- encodings (all leak-free per fold)
HIER_A = 6.0                    # shrinkage strength of the hierarchical encodings
KERN_W = (3, 15)                # neighbouring-distance windows (in distinct Flight Distance values)
PARENT = {"fd": None, "fd_age": "fd", "identity": "fd_age", "flight": "fd"}
KNN_NAMES = ["knn5", "knn20", "knn50", "kd5", "kd50"]
KNN_PRESETS = [dict(fd=3.0, age=1.5, cat=4.0, rate=1.0, delay=0.5),   # 0: route / demographics heavy
               dict(fd=1.0, age=0.5, cat=1.5, rate=2.0, delay=0.3),   # 1: ratings heavy
               dict(rate=1.0, delay=0.3)]                             # 2: ratings + delays only


def te2_names(Fe: Feats) -> list:
    return [f"h_{k}" for k in PARENT if k in Fe.key_names] + [f"kern{w}" for w in KERN_W if "fd" in Fe.key_names]


def hier_kern(Fe: Feats, y: np.ndarray, fit: np.ndarray, app: np.ndarray) -> np.ndarray:
    """Hierarchical-shrinkage TE (child shrunk towards parent) and a kernel TE over neighbouring distances.
    Statistics come from the `fit` rows only; the result is for the `app` rows."""
    names = {nm: j for j, nm in enumerate(Fe.key_names)}
    yf = y[fit].astype(np.float64)
    p0 = yf.mean()

    def stats(code):
        size = int(code.max()) + 1
        return np.bincount(code[fit], weights=yf, minlength=size), np.bincount(code[fit], minlength=size).astype(np.float64)

    est, out = {}, []
    for nm in PARENT:
        if nm not in names:
            continue
        code = Fe.keys[:, names[nm]]
        s, c = stats(code)
        prior = est[PARENT[nm]] if PARENT[nm] in est else np.full(len(app), p0)
        est[nm] = (s[code[app]] + HIER_A * prior) / (c[code[app]] + HIER_A)
        out.append(est[nm])
    if "fd" in names:
        code = Fe.keys[:, names["fd"]]
        s, c = stats(code)
        m = len(Fe.fd_rank)
        ss, cc = np.zeros(m), np.zeros(m)
        ss[Fe.fd_rank], cc[Fe.fd_rank] = s, c   # statistics laid out in sorted-distance order
        cs, ccs = np.concatenate([[0.0], np.cumsum(ss)]), np.concatenate([[0.0], np.cumsum(cc)])
        r = Fe.fd_rank[code[app]]
        for w in KERN_W:
            lo, hi = np.maximum(r - w, 0), np.minimum(r + w + 1, m)
            ws, wc = cs[hi] - cs[lo] - ss[r], ccs[hi] - ccs[lo] - cc[r]  # neighbours only, own route excluded
            out.append((ws + HIER_A * p0) / (wc + HIER_A))
    return np.column_stack(out).astype(np.float32)


def te2_fold(Fe: Feats, y: np.ndarray, i_tr: np.ndarray, i_va: np.ndarray):
    inner = np.random.default_rng(SEED).integers(0, 5, len(i_tr))  # train rows are cross-fitted internally
    tr = np.zeros((len(i_tr), len(te2_names(Fe))), np.float32)
    for f in range(5):
        m = inner == f
        tr[m] = hier_kern(Fe, y, i_tr[~m], i_tr[m])
    return tr, hier_kern(Fe, y, i_tr, i_va), hier_kern(Fe, y, i_tr, np.arange(Fe.n, len(Fe.keys)))


def sk_te_fold(Fe: Feats, y: np.ndarray, i_tr: np.ndarray, i_va: np.ndarray):
    n = Fe.n
    if Fe.keys.shape[1] == 0:
        return tuple(np.zeros((m, 0), np.float32) for m in (len(i_tr), len(i_va), len(Fe.keys) - n))
    enc = TargetEncoder(target_type="binary", smooth="auto", cv=5, shuffle=True, random_state=SEED)
    tr = enc.fit_transform(Fe.keys[i_tr], y[i_tr])
    return tr, enc.transform(Fe.keys[i_va]), enc.transform(Fe.keys[n:])


def make_z(Fe: Feats, preset: int) -> np.ndarray:
    w = KNN_PRESETS[preset]
    return np.hstack([Fe.parts[g] * w[g] for g in w if g in Fe.parts]).astype(np.float32)


def knn_label_feats(Z: np.ndarray, y: np.ndarray, pool: np.ndarray, query: np.ndarray, loo: bool, dev: str,
                    ks=(5, 20, 50)) -> np.ndarray:
    """Mean label of the k nearest pool rows (+ mean neighbour distances). With loo=True, query must be the
    pool itself (same order) and each row's own label is excluded."""
    kmax = max(ks)
    P = torch.as_tensor(Z[pool], device=dev)
    yp = torch.as_tensor(y[pool], dtype=torch.float32, device=dev)
    pp = (P * P).sum(1)
    chunk = max(64, int(1.5e8 // len(pool)))
    out = np.zeros((len(query), len(ks) + 2), np.float32)
    with torch.no_grad():
        for s in range(0, len(query), chunk):
            q = torch.as_tensor(Z[query[s:s + chunk]], device=dev)
            d2 = (q * q).sum(1, keepdim=True) + pp[None, :] - 2.0 * (q @ P.T)
            if loo:
                d2[torch.arange(len(q), device=dev), torch.arange(s, s + len(q), device=dev)] = float("inf")
            d, nb = torch.topk(d2, kmax, dim=1, largest=False)
            lab = yp[nb]
            cols = [lab[:, :k].mean(1) for k in ks]
            cols += [d[:, :5].clamp_min(0).sqrt().mean(1), d.clamp_min(0).sqrt().mean(1)]
            out[s:s + len(q)] = torch.stack(cols, 1).cpu().numpy()
    return out


def knn_fold(Fe: Feats, y: np.ndarray, i_tr: np.ndarray, i_va: np.ndarray, dev: str, preset: int):
    Z = make_z(Fe, preset)
    return (knn_label_feats(Z, y, i_tr, i_tr, True, dev), knn_label_feats(Z, y, i_tr, i_va, False, dev),
            knn_label_feats(Z, y, i_tr, np.arange(Fe.n, len(Z)), False, dev))


ENC_KINDS = ("route", "nne")   # feature blocks that contain the encodings


def set_enc_meta(cfg: Cfg, Fe: Feats, groups: set) -> None:
    """Column names + signature of the encodings (used in cache keys) without computing them."""
    names = []
    if "sk" in groups:
        names += [f"te_{k}" for k in Fe.key_names]
    if "te2" in groups:
        names += te2_names(Fe)
    if "knn" in groups:
        names += KNN_NAMES
    Fe.enc_names = names
    Fe.enc_sig = repr((names, cfg.knn_preset, Fe.use_rdev))


class LazyEnc:
    """Encodings are built on first use, so a re-run where every enc-using member is cached skips them
    (the kNN part alone took ~8 minutes on 5 folds)."""

    def __init__(self, cfg: Cfg, Fe: Feats, y: np.ndarray, folds: list, groups: set):
        self.args, self.val, self.lock = (cfg, Fe, y, folds, groups), None, threading.Lock()
        set_enc_meta(cfg, Fe, groups)

    def __getitem__(self, k: int):
        with self.lock:
            if self.val is None:
                self.val = build_enc(*self.args)
        return self.val[k]


def build_enc(cfg: Cfg, Fe: Feats, y: np.ndarray, folds: list, groups: set) -> list:
    """Per fold: (train, valid, test) encoding matrices; column names are in Fe.enc_names."""
    set_enc_meta(cfg, Fe, groups)
    names = Fe.enc_names
    n_te = len(Fe.keys) - Fe.n

    def one(item):
        k, (i_tr, i_va) = item
        parts = []
        if "sk" in groups:
            parts.append(sk_te_fold(Fe, y, i_tr, i_va))
        if "te2" in groups:
            parts.append(te2_fold(Fe, y, i_tr, i_va))
        if "knn" in groups:
            parts.append(knn_fold(Fe, y, i_tr, i_va, gpu(cfg, k % max(cfg.ngpu, 1)), cfg.knn_preset))
        if not parts:
            return tuple(np.zeros((m, 0), np.float32) for m in (len(i_tr), len(i_va), n_te))
        return tuple(np.hstack([p[j] for p in parts]).astype(np.float32) for j in range(3))

    with ThreadPoolExecutor(3) as ex:
        out = list(ex.map(one, enumerate(folds)))
    log(f"encodings ready for {len(folds)} folds: {len(names)} columns {names}")
    return out


def block(kind: str, Fe: Feats, rows: np.ndarray, enc: np.ndarray) -> np.ndarray:
    B, U = Fe.base[rows], Fe.route[rows]
    R = [Fe.rdev[rows]] if Fe.use_rdev else []
    G = [Fe.og[rows]] if Fe.use_og else []   # original-dataset features (see attach_og)
    if kind == "base":
        return np.hstack([B, *G])
    if kind == "route":   # base + route context (+ rating deviations) + original-data + encodings
        return np.hstack([B, U, *R, *G, enc])
    if kind == "nn":      # last len(vocabs) columns are the embedding ids
        return np.hstack([B, U, *R, *G, Fe.ids[rows].astype(np.float32)])
    if kind == "nne":     # same, plus the encodings (as numeric features)
        return np.hstack([B, U, *R, *G, enc, Fe.ids[rows].astype(np.float32)])
    if kind == "cat":     # last column is Flight Distance as a native categorical
        return np.hstack([B, *G, Fe.fd_code[rows, None].astype(np.float32)])
    if kind == "rmlp":    # base + route context + Flight Distance categorical (last column)
        return np.hstack([B, U, *G, Fe.fd_code[rows, None].astype(np.float32)])
    raise ValueError(kind)


# --------------------------------------------------------------------------- neural nets
if torch is not None:
    BINS = 16

    class PLE(nn.Module):
        """Piecewise-linear encoding on quantile bins: tree-like threshold features for the MLP."""

        def __init__(self, X: np.ndarray):
            super().__init__()
            e = np.quantile(X, np.linspace(0, 1, BINS + 1), axis=0).T.astype(np.float32)
            self.register_buffer("lo", torch.tensor(e[:, :-1]))
            self.register_buffer("w", torch.tensor(np.maximum(e[:, 1:] - e[:, :-1], 1e-6)))

        def forward(self, x):
            return ((x[:, :, None] - self.lo) / self.w).clamp(0, 1).flatten(1)

    class BELinear(nn.Module):
        """BatchEnsemble linear layer (TabM): one shared weight, per-member rank-1 scalings r, s and bias b."""

        def __init__(self, d_in: int, d_out: int, k: int, first: bool = False):
            super().__init__()
            self.lin = nn.Linear(d_in, d_out, bias=False)
            r = torch.ones(k, d_in)
            if first and k > 1:  # random signs in the first layer give the members different views
                r = torch.randint(0, 2, (k, d_in)).float() * 2 - 1
            self.r, self.s = nn.Parameter(r), nn.Parameter(torch.ones(k, d_out))
            self.b = nn.Parameter(torch.zeros(k, d_out))

        def forward(self, x):  # x: (batch, k, d_in)
            return self.lin(x * self.r) * self.s + self.b

    class TabMNet(nn.Module):
        """PLE numeric features + id embeddings -> k-member BatchEnsemble MLP. Returns (batch, k) logits."""

        def __init__(self, Xnum: np.ndarray, vocabs: list, width: int, depth: int, drop: float, k: int, emb: int):
            super().__init__()
            self.k, self.nid = k, len(vocabs)
            self.ple = PLE(Xnum)
            self.embs = nn.ModuleList([nn.Embedding(v, emb) for v in vocabs])
            dims = [Xnum.shape[1] * BINS + emb * self.nid] + [width] * depth
            self.layers = nn.ModuleList([BELinear(dims[i], dims[i + 1], k, first=i == 0) for i in range(depth)])
            self.drop = nn.Dropout(drop)
            self.head = BELinear(width, 1, k)

        def forward(self, x):
            m = x.shape[1] - self.nid
            z = torch.cat([self.ple(x[:, :m])] + [e(x[:, m + j].long()) for j, e in enumerate(self.embs)], 1)
            h = z.unsqueeze(1).expand(-1, self.k, -1)
            for layer in self.layers:
                h = self.drop(torch.relu(layer(h)))
            return self.head(h).squeeze(-1)

    def pair_loss(s, t):
        """Pairwise logistic loss over all positive/negative pairs in the batch (a smooth AUC surrogate)."""
        pos, neg = s[t > 0.5], s[t < 0.5]
        if len(pos) == 0 or len(neg) == 0:
            return s.sum() * 0.0
        return F.softplus(-(pos[:, None] - neg[None, :])).mean()

    @torch.no_grad()
    def logits_of(nets: list, X, chunk: int = 32768) -> np.ndarray:
        outs = []
        for i in range(0, len(X), chunk):
            xb = X[i:i + chunk]
            outs.append(torch.stack([net(xb).mean(1) for net in nets]).mean(0))
        return torch.cat(outs).float().cpu().numpy()

    def train_tabm(net, X, y, Xv, yv, epochs: int, batch: int, lr: float, wd: float, auc_w: float,
                   ema_decay: float = 0.998, verbose: bool = False) -> None:
        """One-cycle AdamW; weights are replaced by their EMA at the end. No best-epoch selection."""
        steps = max(len(X) // batch, 1)
        params = list(net.parameters())
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * steps, pct_start=0.1)
        ema, t_step = [p.detach().clone() for p in params], 0
        for ep in range(epochs):
            net.train()
            perm = torch.randperm(len(X), device=X.device)
            for i in range(steps):
                idx = perm[i * batch:(i + 1) * batch]
                out, t = net(X[idx]), y[idx]
                loss = F.binary_cross_entropy_with_logits(out, t[:, None].expand_as(out))
                if auc_w > 0:
                    loss = loss + auc_w * pair_loss(out.mean(1), t)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                sched.step()
                t_step += 1
                dec = min(ema_decay, (1 + t_step) / (10 + t_step))  # warm-up: EMA must not stay near the init
                with torch.no_grad():
                    torch._foreach_mul_(ema, dec)
                    torch._foreach_add_(ema, params, alpha=1 - dec)
            if verbose and ((ep + 1) % 5 == 0 or ep + 1 == epochs):
                net.eval()
                log(f"    epoch {ep + 1}/{epochs} val AUC (raw weights, for monitoring only) "
                    f"{roc_auc_score(yv, logits_of([net], Xv)):.5f}")
        with torch.no_grad():
            for p, e in zip(params, ema):
                p.copy_(e)
        net.eval()


# --------------------------------------------------------------------------- members
def fit_xgb(cfg: Cfg, tr, va, aux: dict, dev: int, depth: int, colsample: float, seed: int) -> Callable:
    params = dict(objective="binary:logistic", eval_metric="auc", eta=0.03, max_depth=depth, subsample=0.8,
                  colsample_bytree=colsample, min_child_weight=5, reg_lambda=2.0, tree_method="hist",
                  device=gpu(cfg, dev), seed=seed)
    dtr = xgb.QuantileDMatrix(*tr)
    dva = xgb.QuantileDMatrix(*va, ref=dtr)
    b = xgb.train(params, dtr, cfg.rounds, evals=[(dva, "va")], early_stopping_rounds=200, verbose_eval=False)
    r = (0, b.best_iteration + 1)
    return lambda X: b.predict(xgb.DMatrix(X), output_margin=True, iteration_range=r)


def qid_blocks(n: int, size: int = 1024) -> np.ndarray:
    """Consecutive blocks of `size` rows act as query groups. One giant group made XGBoost's group-wise AUC metric
    overflow int32 on the GPU (observed on a 350k-row validation fold)."""
    return (np.arange(n) // size).astype(np.int32)


def fit_xgb_rank(cfg: Cfg, tr, va, aux: dict, dev: int, depth: int, colsample: float, seed: int) -> Callable:
    """Pairwise-ranking objective inside blocks of 1024 rows: optimises the ordering (AUC) directly."""
    params = dict(objective="rank:pairwise", eval_metric="auc", eta=0.03, max_depth=depth, subsample=0.8,
                  colsample_bytree=colsample, min_child_weight=5, reg_lambda=2.0, tree_method="hist",
                  device=gpu(cfg, dev), seed=seed, lambdarank_pair_method="mean", lambdarank_num_pair_per_sample=8)
    dtr = xgb.QuantileDMatrix(tr[0], tr[1], qid=qid_blocks(len(tr[1])))
    dva = xgb.QuantileDMatrix(va[0], va[1], qid=qid_blocks(len(va[1])), ref=dtr)
    b = xgb.train(params, dtr, cfg.rounds, evals=[(dva, "va")], early_stopping_rounds=200, verbose_eval=False)
    r = (0, b.best_iteration + 1)
    return lambda X: b.predict(xgb.DMatrix(X), output_margin=True, iteration_range=r)


def as_frame(X: np.ndarray, cat_idx: list) -> pd.DataFrame:
    df = pd.DataFrame(X)
    for i in cat_idx:
        df[i] = df[i].astype(np.int64)
    return df


def fit_cat(cfg: Cfg, tr, va, aux: dict, dev: int, depth: int, lr: float, native: bool,
            extra: dict = None) -> Callable:
    cat_idx = aux["cat_idx"] if native else None
    prep = (lambda X: as_frame(X, cat_idx)) if native else (lambda X: X)
    m = cb.CatBoostClassifier(iterations=cfg.rounds, learning_rate=lr, depth=depth, l2_leaf_reg=3,
                              early_stopping_rounds=200, random_seed=SEED, verbose=0, cat_features=cat_idx,
                              **cat_device(cfg, dev), **(extra or {}))
    m.fit(prep(tr[0]), tr[1], eval_set=(prep(va[0]), va[1]))
    return lambda X: m.predict(prep(X), prediction_type="RawFormulaVal")


def fit_lgb(cfg: Cfg, tr, va, aux: dict, leaves: int, colsample: float, seed: int, lr: float = 0.04,
            extra: dict = None) -> Callable:
    params = dict(objective="binary", learning_rate=lr, num_leaves=leaves, max_bin=255, min_child_samples=40,
                  feature_fraction=colsample, bagging_fraction=0.8, bagging_freq=1, lambda_l2=2.0,
                  num_threads=CPU_THREADS, verbose=-1, seed=seed, **(extra or {}))
    dtr = lgb.Dataset(*tr)
    dva = lgb.Dataset(*va, reference=dtr)
    m = lgb.train(params, dtr, cfg.rounds, valid_sets=[dva], callbacks=[lgb.early_stopping(200, verbose=False)])
    return lambda X: m.predict(X, raw_score=True, num_iteration=m.best_iteration)


def fit_tabm(cfg: Cfg, tr, va, aux: dict, dev: int, width: int, depth: int, k: int, seed: int,
             drop: float = 0.1, epochs: int = 40, batch: int = 1024, lr: float = 2e-3, wd: float = 1e-4,
             emb: int = 16, auc_w: float = 0.0) -> Callable:
    device, nid = gpu(cfg, dev), len(aux["vocabs"])
    X = torch.as_tensor(tr[0], device=device)
    y = torch.as_tensor(tr[1], dtype=torch.float32, device=device)
    Xv = torch.as_tensor(va[0], device=device)
    nets = []
    for s in range(cfg.seeds):
        torch.manual_seed(seed + 100 * s)
        net = TabMNet(tr[0][:, :-nid], aux["vocabs"], width, depth, drop, k, emb).to(device)
        train_tabm(net, X, y, Xv, va[1], cfg.nn_epochs if cfg.quick else epochs, batch, lr, wd, auc_w,
                   verbose=cfg.stage == "bench")
        nets.append(net)

    def predict(Xn: np.ndarray) -> np.ndarray:
        outs = []
        for i in range(0, len(Xn), 65536):
            outs.append(logits_of(nets, torch.as_tensor(Xn[i:i + 65536], device=device)))
        return np.concatenate(outs)
    return predict


def pk_model(cls, **kw):
    """Build a pytabkit model, dropping keyword arguments this pytabkit version does not know."""
    params = inspect.signature(cls.__init__).parameters
    if any(p.kind == p.VAR_KEYWORD for p in params.values()):
        return cls(**kw)
    ok = {k: v for k, v in kw.items() if k in params}
    if len(ok) < len(kw):
        log(f"{cls.__name__}: ignoring unsupported arguments {sorted(set(kw) - set(ok))}")
    return cls(**ok)


def fit_pk(cfg: Cfg, tr, va, aux: dict, dev: int, model: str, seed: int, epochs: int, batch: int,
           **kw) -> Callable:
    """pytabkit RealMLP-TD / TabM-D. Needs `pip install pytabkit` (and internet in the Kaggle session)."""
    import pytabkit
    cat_idx = aux["cat_idx"]
    names = [f"c{i}" for i in range(tr[0].shape[1])]

    def frame(X: np.ndarray) -> pd.DataFrame:
        df = pd.DataFrame(X, columns=names)
        for i in cat_idx:
            df[names[i]] = df[names[i]].astype(np.int64)
        return df

    m = pk_model(getattr(pytabkit, model), device=gpu(cfg, dev), random_state=seed, n_cv=1, n_refit=0,
                 n_epochs=cfg.nn_epochs if cfg.quick else epochs, batch_size=batch,
                 val_metric_name="cross_entropy", **kw)
    m.fit(frame(tr[0]), tr[1], X_val=frame(va[0]), y_val=va[1], cat_col_names=[names[i] for i in cat_idx])

    def predict(X: np.ndarray) -> np.ndarray:
        p = np.clip(m.predict_proba(frame(X))[:, 1], 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))
    return predict


@dataclass
class Member:
    name: str
    lane: str   # members of a lane run one after another; lanes run in parallel
    kind: str   # feature block, see block()
    fit: Callable
    full: bool = True  # False: only runs in --stage bench (or when named with --only)


def make_members() -> list:
    ms = []
    if torch is not None:  # neural nets first: they are the main lever and their fold-1 AUC is the early signal
        ms += [
            Member("tabm_a", "gpu0", "nn", partial(fit_tabm, dev=0, width=512, depth=3, k=8, seed=1, epochs=8), False),
            Member("tabm_te", "gpu1", "nne", partial(fit_tabm, dev=1, width=512, depth=3, k=8, seed=2, epochs=8), False),
            Member("realmlp_long", "gpu0", "rmlp", partial(fit_pk, dev=0, model="RealMLP_TD_Classifier", seed=1,
                                                           epochs=64, batch=512, hidden_sizes=[256, 256, 256])),
            # tabm_pk: 30 epochs took 3118 s per fold and scored 0.946 on fold 1 (it overfits late) -> bench-only
            Member("tabm_pk", "gpu1", "rmlp", partial(fit_pk, dev=1, model="TabM_D_Classifier", seed=2,
                                                      epochs=30, batch=1024), False),
            # bench-only variants (full=False): stronger regularisation, longer schedule, one head, AUC loss
            Member("tabm_reg", "gpu0", "nn", partial(fit_tabm, dev=0, width=512, depth=3, k=8, seed=6, epochs=8,
                                                     drop=0.3, wd=3e-2, emb=8)),
            # architecture / loss variants of tabm_reg for stack diversity (untested on real data)
            Member("tabm_v1", "gpu0", "nn", partial(fit_tabm, dev=0, width=512, depth=4, k=8, seed=11, epochs=8,
                                                    drop=0.3, wd=3e-2, emb=8), False),
            Member("tabm_v2", "gpu1", "nn", partial(fit_tabm, dev=1, width=384, depth=2, k=16, seed=12, epochs=10,
                                                    drop=0.2, wd=3e-2, emb=8), False),
            Member("tabm_v3", "gpu0", "nn", partial(fit_tabm, dev=0, width=512, depth=3, k=8, seed=13, epochs=8,
                                                    drop=0.3, wd=3e-2, emb=8, auc_w=0.3), False),
            Member("tabm_reg16", "gpu1", "nn", partial(fit_tabm, dev=1, width=512, depth=3, k=8, seed=7, epochs=16,
                                                       drop=0.3, wd=3e-2, emb=8), False),
            Member("mlp_k1", "gpu1", "nn", partial(fit_tabm, dev=1, width=512, depth=3, k=1, seed=3, epochs=8),
                   False),
            Member("tabm_auc", "gpu0", "nn", partial(fit_tabm, dev=0, width=512, depth=3, k=8, seed=4, epochs=8,
                                                     auc_w=0.3), False),
            Member("realmlp_short", "gpu0", "rmlp", partial(fit_pk, dev=0, model="RealMLP_TD_Classifier", seed=2,
                                                            epochs=24, batch=1024, hidden_sizes=[256, 256, 256]),
                   False),
        ]
    ms += [
        Member("cat_native", "gpu0", "cat", partial(fit_cat, dev=0, depth=8, lr=0.06, native=True)),
        Member("xgb_deep", "gpu1", "route", partial(fit_xgb, dev=1, depth=9, colsample=0.5, seed=3)),
        # xgb_rank: -0.0014 AUC per fold vs xgb_deep, stack weight 0, raw margins not comparable across folds
        Member("xgb_rank", "gpu0", "route", partial(fit_xgb_rank, dev=0, depth=8, colsample=0.5, seed=4), False),
        Member("cat_loss", "gpu1", "route", partial(fit_cat, dev=1, depth=10, lr=0.06, native=False,
                                                    extra=dict(grow_policy="Lossguide", max_leaves=96,
                                                               min_data_in_leaf=30))),
        Member("lgb_route", "cpu", "route", partial(fit_lgb, leaves=127, colsample=0.6, seed=2)),
        Member("lgb_base", "cpu", "base", partial(fit_lgb, leaves=63, colsample=0.7, seed=1)),
    ]
    return ms


def member_tag(cfg: Cfg, m: Member, Fe: Feats, folds: list, legacy: bool = False) -> str:
    """Cache key: changes whenever the member's inputs or settings change. Members that do not see the
    encodings no longer depend on them (legacy=True reproduces the older key, so old cache files still load)."""
    probe = block(m.kind, Fe, np.arange(2), np.zeros((2, len(Fe.enc_names)), np.float32))
    sig = Fe.enc_sig if (legacy or m.kind in ENC_KINDS) else ""
    spec = (FEATURE_VERSION, m.kind, probe.shape[1], sig, m.fit.func.__name__,
            sorted(m.fit.keywords.items(), key=str), cfg.rounds, cfg.quick,
            cfg.seeds if (legacy or m.kind in ("nn", "nne")) else 1, len(folds), Fe.n,
            len(Fe.base)) + ((Fe.og_sig,) if Fe.use_og else ())
    return hashlib.md5(repr(spec).encode()).hexdigest()[:8]


def aux_for(Fe: Feats, Xtr: np.ndarray) -> dict:
    return dict(vocabs=Fe.vocabs, cat_idx=Fe.cat_idx + [Xtr.shape[1] - 1])


def run_member(cfg: Cfg, m: Member, Fe: Feats, y: np.ndarray, folds: list, enc: list):
    cache = Path(cfg.out) / "cache"
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / f"{m.name}_{member_tag(cfg, m, Fe, folds)}.npz"
    for old in (path, cache / f"{m.name}_{member_tag(cfg, m, Fe, folds, legacy=True)}.npz"):
        if old.exists():  # resume after a session timeout
            z = np.load(old)
            log(f"{m.name}: loaded from cache (OOF AUC {roc_auc_score(y, z['oof']):.5f})")
            return z["oof"], z["te"]

    n, n_all = Fe.n, len(Fe.base)
    rows_te = np.arange(n, n_all)
    oof, te = np.zeros(n, np.float32), np.zeros(n_all - n, np.float32)
    for k, (i_tr, i_va) in enumerate(folds):
        e_tr, e_va, e_te = enc[k] if m.kind in ENC_KINDS else (None, None, None)
        Xtr, Xva, Xte = (block(m.kind, Fe, r, e) for r, e in ((i_tr, e_tr), (i_va, e_va), (rows_te, e_te)))
        predict = m.fit(cfg, (Xtr, y[i_tr]), (Xva, y[i_va]), aux_for(Fe, Xtr))
        oof[i_va] = predict(Xva)
        te += predict(Xte) / len(folds)
        log(f"{m.name:12s} fold {k + 1}/{len(folds)}  AUC {roc_auc_score(y[i_va], oof[i_va]):.5f}")
        del Xtr, Xva, Xte, predict
    log(f"{m.name}: OOF AUC {roc_auc_score(y, oof):.5f}")
    np.savez(path, oof=oof, te=te)
    return oof, te


def run_lanes(members: list, work: Callable) -> None:
    lanes = {}
    for m in members:
        lanes.setdefault(m.lane, []).append(m)

    def run_lane(items):
        for m in items:
            try:
                work(m)
            except Exception:
                print(f"FAILED member {m.name}")
                traceback.print_exc()

    with ThreadPoolExecutor(max(len(lanes), 1)) as ex:
        list(ex.map(run_lane, lanes.values()))


# --------------------------------------------------------------------------- stacking
def fit_stack(Z: np.ndarray, y: np.ndarray):
    """Non-negative logistic stacker on member logits (weights >= 0 stops collinear members cancelling)."""
    Z = Z.astype(np.float64)

    def fun(p):
        s = Z @ p[:-1] + p[-1]
        r = expit(s) - y
        return np.mean(np.logaddexp(0, s) - y * s), np.append(Z.T @ r, r.sum()) / len(y)

    p0 = np.append(np.full(Z.shape[1], 1 / Z.shape[1]), 0.0)
    bounds = [(0, None)] * Z.shape[1] + [(None, None)]
    p = minimize(fun, p0, jac=True, method="L-BFGS-B", bounds=bounds).x
    return p[:-1], p[-1]


def stack_cv_auc(Z: np.ndarray, y: np.ndarray) -> float:
    """Honest stacker estimate: fit the weights on one half of the rows, score on the other half."""
    idx = np.random.default_rng(SEED).permutation(len(y))
    a, b = idx[: len(y) // 2], idx[len(y) // 2:]
    scores = []
    for tr, va in ((a, b), (b, a)):
        w, c = fit_stack(Z[tr], y[tr])
        scores.append(roc_auc_score(y[va], Z[va] @ w + c))
    return float(np.mean(scores))


def stack2_lgb(Z, Zte, meta, meta_te, y, folds):
    """LightGBM on member logits + meta columns, scored fold-wise next to the linear stacker on the SAME folds.
    NOTE: member OOF values were produced with these same folds, so both scores are slightly optimistic;
    the comparison between them is the useful number."""
    X, Xte = np.hstack([Z, meta]), np.hstack([Zte, meta_te])
    params = dict(objective="binary", learning_rate=0.03, num_leaves=15, min_child_samples=200, feature_fraction=0.8,
                  bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, num_threads=CPU_THREADS, verbose=-1,
                  seed=SEED)
    oof_g, oof_l, te_g = np.zeros(len(y)), np.zeros(len(y)), np.zeros(len(Xte))
    for tr, va in folds:
        w, b = fit_stack(Z[tr], y[tr])
        oof_l[va] = Z[va] @ w + b
        d = lgb.Dataset(X[tr], y[tr])
        m = lgb.train(params, d, 3000, valid_sets=[lgb.Dataset(X[va], y[va], reference=d)],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        oof_g[va] = m.predict(X[va], raw_score=True, num_iteration=m.best_iteration)
        te_g += m.predict(Xte, raw_score=True, num_iteration=m.best_iteration) / len(folds)
    return oof_l, oof_g, te_g


# --------------------------------------------------------------------------- stages: diag / ablate / bench
def auc_or_nan(y, s) -> float:
    return float(roc_auc_score(y, s)) if len(np.unique(y)) > 1 else float("nan")


def diag(cfg: Cfg, Fe: Feats, y: np.ndarray) -> None:
    """Cheap data diagnostics. (1) How much label signal do rows that share a key carry (leave-one-out group
    mean as the predictor of the row's own label)? (2) How informative are nearest neighbours, per retrieval space?"""
    n = len(y)
    rows = []
    for j, nm in enumerate(Fe.key_names):
        size = int(Fe.keys[:, j].max()) + 1
        code = Fe.keys[:n, j]
        cnt = np.bincount(code, minlength=size)[code]
        s = np.bincount(code, weights=y, minlength=size)[code]
        m, sm = cnt >= 2, (cnt >= 2) & (cnt <= 5)
        rows.append(dict(key=nm, distinct=int((np.bincount(code, minlength=size) > 0).sum()),
                         share_in_groups_ge2=m.mean(), median_group=float(np.median(cnt)),
                         auc_loo=auc_or_nan(y[m], (s[m] - y[m]) / (cnt[m] - 1)),
                         auc_loo_groups_2to5=auc_or_nan(y[sm], (s[sm] - y[sm]) / (cnt[sm] - 1))))
    print("\nLABEL SIGNAL INSIDE KEY GROUPS (leave-one-out group mean vs own label; 0.5 = none)\n"
          + pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    if torch is None:
        return
    nsub = 200_000 if cfg.ngpu else 20_000
    sub = np.sort(np.random.default_rng(SEED).choice(n, min(nsub, n), replace=False))
    res = []
    for p in range(len(KNN_PRESETS)):
        Z = make_z(Fe, p)
        f = knn_label_feats(Z, y, sub, sub, True, gpu(cfg, 0))
        res.append(dict(preset=p, weights=str(KNN_PRESETS[p]), auc_knn5=auc_or_nan(y[sub], f[:, 0]),
                        auc_knn20=auc_or_nan(y[sub], f[:, 1]), auc_knn50=auc_or_nan(y[sub], f[:, 2])))
    print(f"\nkNN LABEL FEATURE ALONE (leave-one-out, {len(sub):,}-row subsample so absolute AUCs understate "
          f"the full-data value; compare presets with each other)\n"
          + pd.DataFrame(res).to_string(index=False, float_format=lambda v: f"{v:.4f}"))


def xgb_quick(Xtr, ytr, Xva, yva, dev: str, seed: int) -> float:
    params = dict(objective="binary:logistic", eval_metric="auc", eta=0.08, max_depth=7, subsample=0.8,
                  colsample_bytree=0.6, min_child_weight=5, reg_lambda=2.0, tree_method="hist", device=dev,
                  seed=seed)
    dtr = xgb.QuantileDMatrix(Xtr, ytr)
    dva = xgb.QuantileDMatrix(Xva, yva, ref=dtr)
    b = xgb.train(params, dtr, 100 if dtr.num_row() < 50_000 else 3000, evals=[(dva, "va")],
                  early_stopping_rounds=100, verbose_eval=False)
    return float(b.best_score)


def ablate(cfg: Cfg, Fe: Feats, y: np.ndarray, folds: list, enc: list) -> None:
    """Fold-1 feature ablation with a fast XGBoost: which route encodings carry signal?"""
    i_tr, i_va = folds[0]
    e_tr, e_va, _ = enc[0]
    names = Fe.enc_names
    pos = {c: j for j, c in enumerate(names)}
    sk = [f"te_{k}" for k in Fe.key_names]
    hk = [c for c in names if c.startswith(("h_", "kern"))]
    kn = [c for c in names if c.startswith(("knn", "kd"))]

    ogpos = {c: j for j, c in enumerate(Fe.og_names)}

    def build(rows, e, ctx, cols, rdev, ogc=()):
        parts = [Fe.base[rows]] + ([Fe.route[rows]] if ctx else []) + ([Fe.rdev[rows]] if rdev else [])
        if ogc:
            parts.append(Fe.og[rows][:, [ogpos[c] for c in ogc]])
        return np.hstack(parts + [e[:, [pos[c] for c in cols]]])

    jobs = [("base only (no route features)", False, [], False), ("base + route context", True, [], False),
            ("v3 route block (context + all 7 TE)", True, sk, False)]
    jobs += [(f"v3 minus {c}", True, [x for x in sk if x != c], False) for c in sk]
    jobs += [("v3 + hierarchical/kernel TE", True, sk + hk, False), ("v3 + rating deviations", True, sk, True),
             ("v3 + hier/kernel TE + deviations", True, sk + hk, True)]
    if kn:
        jobs += [("v3 + kNN labels", True, sk + kn, False), ("v3 + everything new", True, sk + hk + kn, True)]
    if Fe.og is not None:   # original-dataset features (the competition rows were generated from that data)
        o2 = [c for c in ("og_xgb", "og_lgb") if c in ogpos]
        o5 = [c for c in Fe.og_names if c.startswith("og_")]
        om = [c for c in Fe.og_names if c.startswith("om_")]
        ok = [c for c in Fe.og_names if c.startswith("ogk_")]
        jobs += [("base + original-data predictions (no route)", False, [], False, o2),
                 ("v3 + og_xgb", True, sk, False, o2[:1]), ("v3 + og_xgb + og_lgb", True, sk, False, o2),
                 ("v3 + og predictions + route aggregates", True, sk, False, o5),
                 ("context + og predictions + aggregates (no TE)", True, [], False, o5)]
        if om:
            jobs.append(("v3 + og5 + original-data target encodings", True, sk, False, o5 + om))
        if ok:
            jobs.append(("v3 + og5 + original-data kNN", True, sk, False, o5 + ok))
        jobs.append(("v3 + all original-data features", True, sk, False, o5 + om + ok))
    def run(item):
        (nm, ctx, cols, rdev, *rest), idx = item
        ogc = rest[0] if rest else ()
        Xtr, Xva = build(i_tr, e_tr, ctx, cols, rdev, ogc), build(i_va, e_va, ctx, cols, rdev, ogc)
        aucs = [xgb_quick(Xtr, y[i_tr], Xva, y[i_va], gpu(cfg, idx % max(cfg.ngpu, 1)), SEED + s) for s in range(cfg.abl_seeds)]
        log(f"{nm:42s} AUC {np.mean(aucs):.5f}  (n_features={Xtr.shape[1]})")
        return nm, float(np.mean(aucs)), float(np.ptp(aucs)) if len(aucs) > 1 else 0.0

    with ThreadPoolExecutor(max(cfg.ngpu, 1)) as ex:
        res = list(ex.map(run, [(j, i) for i, j in enumerate(jobs)]))
    ref = dict((r[0], r[1]) for r in res)["v3 route block (context + all 7 TE)"]
    tab = pd.DataFrame([dict(config=r[0], fold1_auc=r[1], delta_vs_v3=r[1] - ref, seed_spread=r[2]) for r in res])
    print("\nFOLD-1 ABLATION (fast XGB; deltas smaller than ~2x seed_spread are noise)\n"
          + tab.to_string(index=False, float_format=lambda v: f"{v:.5f}"))


def bench(cfg: Cfg, members: list, Fe: Feats, y: np.ndarray, folds: list, enc: list) -> None:
    i_tr, i_va = folds[0]
    res = {}

    def work(m: Member):
        e_tr, e_va, _ = enc[0] if m.kind in ENC_KINDS else (None, None, None)
        Xtr, Xva = block(m.kind, Fe, i_tr, e_tr), block(m.kind, Fe, i_va, e_va)
        t = time.time()
        predict = m.fit(cfg, (Xtr, y[i_tr]), (Xva, y[i_va]), aux_for(Fe, Xtr))
        auc = roc_auc_score(y[i_va], predict(Xva))
        res[m.name] = (auc, time.time() - t)
        log(f"{m.name:14s} fold-1 AUC {auc:.5f}  ({time.time() - t:.0f}s)")

    run_lanes(members, work)
    print("\nFOLD-1 BENCHMARK\n" + pd.DataFrame([dict(member=k, fold1_auc=v[0], seconds=v[1])
                                                 for k, v in res.items()]).to_string(
        index=False, float_format=lambda v: f"{v:.5f}" if v < 10 else f"{v:.0f}"))
    print(REF_FOLD1)




def adversarial(cfg: Cfg, Fe: Feats) -> None:
    """Can a model tell train rows from test rows? AUC ~0.5 = same distribution. A feature set that scores clearly
    above 0.5 behaves differently on test, which would explain an OOF that does not carry over to the public LB."""
    n, N = Fe.n, len(Fe.base)
    lab = np.r_[np.zeros(n), np.ones(N - n)]
    sets = {"base columns": Fe.base, "base + route context": np.hstack([Fe.base, Fe.route])}
    if Fe.og is not None:
        sets["original-data features only"] = Fe.og
        sets["base + route + original-data"] = np.hstack([Fe.base, Fe.route, Fe.og])
    rows = []
    for i, (name, M) in enumerate(sets.items()):
        oof = np.zeros(N)
        for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(M, lab):
            params = dict(objective="binary:logistic", eval_metric="auc", eta=0.1, max_depth=6, subsample=0.8,
                          colsample_bytree=0.7, tree_method="hist", device=gpu(cfg, i % max(cfg.ngpu, 1)), seed=SEED)
            da = xgb.QuantileDMatrix(M[a], lab[a])
            bst = xgb.train(params, da, 60 if cfg.quick else 200, verbose_eval=False)
            oof[b] = bst.predict(xgb.DMatrix(M[b]))
        rows.append(dict(feature_set=name, train_vs_test_auc=roc_auc_score(lab, oof)))
        log(f"adversarial {name}: {rows[-1]['train_vs_test_auc']:.5f}")
    print("\nTRAIN vs TEST (0.5 = indistinguishable)\n" + pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    if Fe.og is not None:
        q = pd.DataFrame({"split": ["train"] * n + ["test"] * (N - n), "og_xgb": Fe.og[:, 0]})
        print("\nog_xgb logit by split\n" + q.groupby("split")["og_xgb"].describe().round(3).to_string())


# --------------------------------------------------------------------------- stage boost: NN margin + pseudo-labels
def nn_job(cfg: Cfg, m: Member, Fe: Feats, y: np.ndarray, folds: list, fold_id: np.ndarray, j: int, k: int, dev: int):
    """Train the net without folds {j, k} and predict fold j (margin for the rows of fold j inside outer fold k).
    j = -1: train without fold k only, predict fold k and the test rows (the outer model)."""
    cache = Path(cfg.out) / "cache"
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / f"boost_{m.name}_{member_tag(cfg, m, Fe, folds)}_{j}_{k}.npz"
    if path.exists():
        return dict(np.load(path))
    out_rows = np.flatnonzero(~np.isin(fold_id, [k] if j < 0 else [j, k]))
    pred_rows = folds[k][1] if j < 0 else folds[j][1]
    Xtr, Xp = block(m.kind, Fe, out_rows, None), block(m.kind, Fe, pred_rows, None)
    predict = m.fit.func(cfg, (Xtr, y[out_rows]), (Xp, y[pred_rows]), aux_for(Fe, Xtr), **{**m.fit.keywords, "dev": dev})
    res = dict(p=predict(Xp))
    if j < 0:
        res["te"] = predict(block(m.kind, Fe, np.arange(Fe.n, len(Fe.base)), None))
    np.savez(path, **res)
    log(f"nested NN job done: outside folds {{{'k' if j < 0 else f'{j},{k}'}}} (k={k})")
    return res


def xgb_boost(cfg: Cfg, Xtr, ytr, Xva, yva, dev: str, depth: int, seed: int, mtr=None, mva=None, wtr=None):
    """XGBoost with optional base margins / sample weights. Returns predict(X, margin) -> logits (margin included)."""
    params = dict(objective="binary:logistic", eval_metric="auc", eta=0.03, max_depth=depth, subsample=0.8,
                  colsample_bytree=0.5, min_child_weight=5, reg_lambda=2.0, tree_method="hist", device=dev, seed=seed)
    dtr = xgb.QuantileDMatrix(Xtr, ytr, base_margin=mtr, weight=wtr)
    dva = xgb.QuantileDMatrix(Xva, yva, base_margin=mva, ref=dtr)
    b = xgb.train(params, dtr, cfg.rounds, evals=[(dva, "va")], early_stopping_rounds=200, verbose_eval=False)
    r = (0, b.best_iteration + 1)
    return lambda X, mg=None: b.predict(xgb.DMatrix(X, base_margin=mg), output_margin=True, iteration_range=r)


def stage_boost(cfg: Cfg, Fe: Feats, y: np.ndarray, folds: list, enc) -> None:
    """(1) boost XGBoost from a NESTED net margin, (2) pseudo-label the test rows with a fold-clean teacher.
    Nothing here sees a validation fold's labels: margins of the training rows come from nets trained without
    that fold, and the teacher of fold k is built only from models trained without fold k."""
    nn_m = next(m for m in make_members() if m.name == cfg.nn_name)
    K, n, N = len(folds), Fe.n, len(Fe.base)
    fold_id = np.empty(n, np.int64)
    for k, (_, va) in enumerate(folds):
        fold_id[va] = k
    jobs = [(j, k) for k in range(K) for j in range(-1, K) if j != k]
    log(f"nested NN '{nn_m.name}': {len(jobs)} trainings ({K} outer + {len(jobs) - K} inner)")
    with ThreadPoolExecutor(max(cfg.ngpu, 1)) as ex:
        outs = list(ex.map(lambda a: nn_job(cfg, nn_m, Fe, y, folds, fold_id, *a[1], a[0] % max(cfg.ngpu, 1)),
                           enumerate(jobs)))
    nn_out = dict(zip(jobs, outs))

    grid = [tuple(float(v) for v in g.split(":")) for g in cfg.pl_grid.split(",") if g]
    half = np.random.default_rng(SEED).integers(0, 2, N - n)   # test halves for cross-fitted pseudo-label students
    depths = (4, 7)
    names = ["nn_outer", "xgb_plain"] + [f"xgb_margin_d{d}" for d in depths] + [f"xgb_pl_w{w}_c{c}" for w, c in grid]
    oof = {nm: np.zeros(n) for nm in names}
    te = {nm: np.zeros(N - n) for nm in names}

    def one_fold(item):
        k, (i_tr, i_va) = item
        dev = gpu(cfg, k % max(cfg.ngpu, 1))
        e_tr, e_va, e_te = enc[k]
        rows_te = np.arange(n, N)
        Xtr, Xva, Xte = (block("route", Fe, r, e) for r, e in ((i_tr, e_tr), (i_va, e_va), (rows_te, e_te)))
        m_tr = np.zeros(len(i_tr))
        for j in range(K):
            if j != k:
                m_tr[np.searchsorted(i_tr, folds[j][1])] = nn_out[(j, k)]["p"]
        m_va, m_te = nn_out[(-1, k)]["p"], nn_out[(-1, k)]["te"]
        res = {"nn_outer": (m_va, m_te)}
        pred = xgb_boost(cfg, Xtr, y[i_tr], Xva, y[i_va], dev, 8, SEED)
        res["xgb_plain"] = (pred(Xva), pred(Xte))
        for d in depths:   # boost from the nested net margin
            pred = xgb_boost(cfg, Xtr, y[i_tr], Xva, y[i_va], dev, d, SEED, m_tr, m_va)
            res[f"xgb_margin_d{d}"] = (pred(Xva, m_va), pred(Xte, m_te))
        teacher = 0.5 * (m_te + res["xgb_plain"][1])          # fold-clean teacher logits for the test rows
        soft = expit(teacher)
        for w, c in grid:                                       # pseudo-label students, cross-fitted over test halves
            keep = np.abs(soft - 0.5) >= c
            v, t = np.zeros(len(i_va)), np.zeros(N - n)
            for h in (0, 1):
                idx = np.flatnonzero((half == h) & keep)
                Xs = np.vstack([Xtr, Xte[idx]])
                ys = np.concatenate([y[i_tr].astype(np.float64), soft[idx]])
                ws = np.concatenate([np.ones(len(i_tr)), np.full(len(idx), w)])
                pred = xgb_boost(cfg, Xs, ys, Xva, y[i_va], dev, 8, SEED + h, wtr=ws)
                v += pred(Xva) / 2
                t[half != h] = pred(Xte[half != h])
            res[f"xgb_pl_w{w}_c{c}"] = (v, t)
        log(f"boost fold {k + 1}/{K}: " + " | ".join(f"{nm} {roc_auc_score(y[i_va], res[nm][0]):.5f}" for nm in names))
        return k, res

    with ThreadPoolExecutor(max(cfg.ngpu, 1)) as ex:
        for k, res in ex.map(one_fold, enumerate(folds)):
            for nm in names:
                oof[nm][folds[k][1]] = res[nm][0]
                te[nm] += res[nm][1] / K

    O, T = np.column_stack([oof[nm] for nm in names]), np.column_stack([te[nm] for nm in names])
    base = [0, 1]  # nn_outer, xgb_plain
    rows = []
    for j, nm in enumerate(names):
        cols = base if j in base else base + [j]
        rows.append(dict(member=nm, oof_auc=roc_auc_score(y, O[:, j]),
                         stack_with_nn_and_plain=stack_cv_auc(O[:, cols], y),
                         gain_over_nn_plain_stack=stack_cv_auc(O[:, cols], y) - stack_cv_auc(O[:, base], y)))
    print("\nBOOST STAGE (held-out stacker estimates; the last column is the added value of that member)\n"
          + pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    np.savez(Path(cfg.out) / "oof_test_logits_boost.npz", names=names, oof=O, te=T, w=np.zeros(len(names)), b=0.0)
    log(f"saved {Path(cfg.out) / 'oof_test_logits_boost.npz'} (importable with --import-npz)")


# --------------------------------------------------------------------------- main
def main() -> None:
    cfg = make_cfg()
    if torch is not None:
        torch.set_num_threads(CPU_THREADS)
    X, y, test, sub, tcol, rating, cat_idx, cat_maps = load(cfg)
    Fe = build_features(X, len(y), rating, cat_idx, cfg)
    folds = list(StratifiedKFold(cfg.folds, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
    if cfg.og:
        attach_og(cfg, X, cat_maps, rating, Fe, y, everything=cfg.stage == "ablate")
    if cfg.stage == "diag":
        diag(cfg, Fe, y)
        return
    if cfg.stage == "adv":
        adversarial(cfg, Fe)
        return

    one = cfg.stage in ("ablate", "bench")
    if cfg.stage == "boost" and (Fe.keys.shape[1] == 0 or len(folds) < 3):
        raise SystemExit("boost stage needs the route encodings and at least 3 folds (not --quick: it forces 2)")
    groups = {"sk"} | ({"te2"} if cfg.te2 or cfg.stage == "ablate" else set())
    if (cfg.knn or cfg.stage == "ablate") and torch is not None:
        groups.add("knn")
    enc = LazyEnc(cfg, Fe, y, folds[:1] if one else folds, groups)
    if cfg.stage == "ablate":
        ablate(cfg, Fe, y, folds, enc)
        return

    if cfg.stage == "boost":
        stage_boost(cfg, Fe, y, folds, enc)
        return

    allm = make_members()
    members = [m for m in allm if (m.name in cfg.only if cfg.only else
                                   (m.kind in ("nn", "nne", "rmlp") if cfg.stage == "bench" else m.full))]
    if cfg.stage == "bench":
        bench(cfg, members, Fe, y, folds, enc)
        return

    results = {}
    run_lanes(members, lambda m: results.__setitem__(m.name, run_member(cfg, m, Fe, y, folds, enc)))
    names = [m.name for m in members if m.name in results]
    oof = np.column_stack([results[n][0] for n in names])
    te = np.column_stack([results[n][1] for n in names])
    for f in cfg.imports:  # members of earlier runs of your own pipeline (same train/test row order)
        try:
            z = np.load(f, allow_pickle=False)
            io, it, inames = z["oof"], z["te"], [f"{Path(f).stem}:{n}" for n in z["names"]]
            assert io.shape[0] == len(y) and it.shape[0] == len(sub), f"row counts differ: {io.shape} {it.shape}"
            oof, te, names = np.hstack([oof, io]), np.hstack([te, it]), names + inames
            log(f"imported {len(inames)} members from {f}")
        except Exception as e:
            print(f"could not import {f}: {e!r}")
    w, b = fit_stack(oof, y)
    stack_auc = roc_auc_score(y, oof @ w + b)

    lost = np.zeros(len(names))  # AUC lost if a member is removed: how much unique signal it carries
    for j in (np.flatnonzero(w > 1e-4) if len(names) > 1 else []):
        keep = [i for i in range(len(names)) if i != j]
        w2, b2 = fit_stack(oof[:, keep], y)
        lost[j] = stack_auc - roc_auc_score(y, oof[:, keep] @ w2 + b2)
    table = pd.DataFrame(dict(member=names, oof_auc=[roc_auc_score(y, oof[:, j]) for j in range(len(names))],
                              stack_weight=w, auc_lost_if_dropped=lost))
    print("\nMEMBERS\n" + table.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    corr = pd.DataFrame(np.atleast_2d(np.corrcoef(oof.T)), index=names, columns=names)
    print("\nOOF logit correlation:\n" + corr.round(3).to_string())
    log(f"best single member {table.oof_auc.max():.5f} | simple mean {roc_auc_score(y, oof.mean(1)):.5f} "
        f"| STACKED OOF AUC {stack_auc:.5f} | stacker held-out estimate {stack_cv_auc(oof, y):.5f}")

    final = te @ w + b
    if cfg.stack2 and len(names) > 1:
        oof_l, oof_g, te_g = stack2_lgb(oof, te, Fe.meta[:Fe.n], Fe.meta[Fe.n:], y, folds)
        blend = 0.5 * (oof_l + oof_g)
        sc = {"linear": roc_auc_score(y, oof_l), "lgbm": roc_auc_score(y, oof_g), "blend": roc_auc_score(y, blend)}
        log("level-2 (same-fold CV, comparable with each other): "
            + " | ".join(f"{k} {v:.5f}" for k, v in sc.items()))
        best = max(sc, key=sc.get)
        if best != "linear" and sc[best] > sc["linear"] + 0.00005:
            final = te_g if best == "lgbm" else 0.5 * (final + te_g)
            log(f"using level-2 '{best}' for the submission")
        else:
            log("keeping the linear stacker (level-2 gain below 0.00005)")

    prob = expit(final)
    idcol = sub.columns[0]
    ids = test[idcol].to_numpy() if idcol in test else sub[idcol].to_numpy()
    out = pd.DataFrame({idcol: ids, tcol: prob})
    assert len(out) == len(sub) and np.isfinite(prob).all() and ((prob >= 0) & (prob <= 1)).all()
    path = Path(cfg.out) / "submission.csv"
    out.to_csv(path, index=False)
    np.savez(Path(cfg.out) / "oof_test_logits.npz", names=names, oof=oof, te=te, w=w, b=b)
    log(f"wrote {path} ({len(out):,} rows, mean prob {prob.mean():.4f} vs train rate {y.mean():.4f})")


if __name__ == "__main__":
    main()


## A. Checks (about 4 min total)
**A1. Is 0.96073 noise?** Draws random public-sized subsets of your own OOF predictions and computes the difference between the original-data stack (B) and the non-original-data stack (A) each time. It measures *sampling* noise only. Needs `run_og5.npz` (made by the `og5` run; if that session is gone, upload `og5/oof_test_logits.npz` as a dataset and point `B_NPZ` at it). `PUBLIC_ROWS` is a guess.

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

A_NPZ = "/kaggle/input/datasets/yukrxfo/airplane/run_v4_tabm.npz"   # stack A: no original data (5-fold)
B_NPZ = "run_og5.npz"                                              # stack B: with original data (5-fold)
PUBLIC_ROWS, DRAWS = 90_000, 600                                   # public-LB size is a guess; change it freely

y = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/train.csv")["satisfaction"] \
      .astype(str).str.lower().isin(["true", "1", "1.0"]).to_numpy().astype(int)
def stacked(path):
    z = np.load(path); return z["oof"] @ z["w"] + z["b"]       # the linear stacker saved with the file
a, b = stacked(A_NPZ), stacked(B_NPZ)
print(f"full OOF AUC  A {roc_auc_score(y, a):.5f}   B {roc_auc_score(y, b):.5f}   B-A {roc_auc_score(y, b) - roc_auc_score(y, a):+.5f}")

rng = np.random.default_rng(0)
d = np.array([roc_auc_score(y[i], b[i]) - roc_auc_score(y[i], a[i])
              for i in (rng.choice(len(y), PUBLIC_ROWS, replace=False) for _ in range(DRAWS))])
print(f"B-A on random {PUBLIC_ROWS:,}-row subsets: mean {d.mean():+.5f}, SD {d.std():.5f}, "
      f"5%-95% range [{np.quantile(d, .05):+.5f}, {np.quantile(d, .95):+.5f}]")
print(f"share of subsets where B scores BELOW A: {(d < 0).mean():.1%}   (observed public LB: B-A = -0.00003)")


**A2. Adversarial check.** A model tries to tell train rows from test rows, using the raw columns, the route features and the original-data features. AUC near 0.5 means they look alike. A clearly higher value for one set would mean that set behaves differently on test, which would explain an OOF gain that never reaches the LB.

In [ ]:
!python s6e10_v4.py --stage adv --og auto

## B. Quick mixed stack (5-fold; about 2 min if `og5/` still exists, else about 21 min)
The original-data members are 0.996-0.999 correlated with each other, so the stack gained only +0.00019. Your earlier members without original data are less correlated with them. This re-stacks both sets with the cached `og5` members and prints a held-out estimate to compare with 0.96118 (original data only) and 0.96099 (none).

In [ ]:
!python s6e10_v4.py --folds 5 --og auto --stack2 --out og5 --only cat_native,xgb_deep,cat_loss,lgb_route,lgb_base,tabm_reg --import-npz /kaggle/input/datasets/yukrxfo/airplane/run_v4_tabm.npz

## C. Final run: 10 folds with the original data
- **C1 (about 45-50 min; guess)** trains the six members on 90% of the rows each and averages 10 models for the test predictions. The copy line reuses the original-data features computed in the 5-fold run (saves about 4 min; harmless if the file is missing).
- Members: `cat_native`, `xgb_deep`, `cat_loss`, `lgb_route`, `lgb_base`, `tabm_reg`. No pytabkit, kNN or `--te2` (none helped).
- If the session dies, re-run the same cell; finished members load from `og10/cache`.
- **C2 (about 3 min)** re-stacks the 10-fold members together with your non-original-data members. Use it only if B showed that mixing helps. Imports must be in `/kaggle/input/...` or the working directory: `run_final10.npz` is the 10-fold no-original-data file from the earlier run; delete it from the command if you no longer have it.

In [ ]:
!mkdir -p og10 && cp og5/og_features_*.npz og10/ 2>/dev/null; true
!python s6e10_v4.py --folds 10 --og auto --stack2 --out og10 --only cat_native,xgb_deep,cat_loss,lgb_route,lgb_base,tabm_reg
!cp og10/submission.csv submission_og10.csv
!cp og10/oof_test_logits.npz run_og10.npz

In [ ]:
# C2: mixed final stack (run after C1; reads the cached og10 members)
!python s6e10_v4.py --folds 10 --og auto --stack2 --out og10 --only cat_native,xgb_deep,cat_loss,lgb_route,lgb_base,tabm_reg --import-npz /kaggle/input/datasets/yukrxfo/airplane/run_v4_tabm.npz,run_final10.npz
!cp og10/submission.csv submission_mixed.csv
!cp submission_mixed.csv submission.csv     # Kaggle reads /kaggle/working/submission.csv; use submission_og10.csv instead if C2 is not better

## D. Choosing the final submissions
- Your public LBs are all within 0.96069-0.96079, while the OOFs span 0.96087-0.96118, so **the public LB cannot rank them**. Decide by held-out OOF (700k rows) rather than by one LB number.
- Kaggle lets you pick two for the private score. Suggested: **(1)** the highest held-out OOF from C1/C2 (A2 should show no train/test difference first); **(2)** your 0.96079 blend of earlier submissions as a hedge.
- Blending submission files cannot be scored by OOF; the stack in C2 can. Prefer C2 over file averaging for anything new.

## What we measured (all done; none of it needs re-running)
| Idea | Result | Verdict |
|---|---|---|
| Route context features | +0.00135 (fast XGB, fold 1) | **keep** |
| The 7 target-encoding columns on top | -0.0001 (noise) | harmless, kept |
| Hierarchical / kernel TE, rating deviations (`--te2`) | 0 to -0.00026 | dropped |
| kNN labels within the competition data (`--knn`) | +0.00008 in a rerun (was +0.00029); no gain at 5 folds | dropped |
| `xgb_rank` | -0.0014 per fold vs `xgb_deep`; weight 0 | dropped |
| `realmlp_long` | OOF 0.95857 vs trees 0.9602; 38 min; 0.00001 lost if dropped | dropped |
| `tabm_pk` | 0.946 on fold 1 after 52 min | dropped |
| Own TabM, 40 epochs | overfits after ~5 epochs | replaced |
| **`tabm_reg`** (8 heads, 8 epochs, dropout 0.3, wd 0.03) | OOF 0.95979 (5-fold) / 0.95995 (10-fold), ~87 s per fold; +0.00012 to the stack | **keep** |
| `tabm_v1`-`v3` variants | fold 1: 0.95983-0.96008 vs 0.96017 | no better |
| 3-seed bagging of `tabm_reg` | member +0.0004, stack +0.00003 | not worth 3x |
| 10 folds (no original data) | members +0.0001-0.0003; stack 0.96103 / 0.96109; LB 0.96075 | small |
| Nested NN-margin boosting, pseudo-labels | +0.00004 to +0.00006 on a 2-member stack | nothing |
| **Original dataset** (models fitted on it, per-route aggregates) | members +0.0004 to +0.0016; stack OOF 0.96118 (+0.00019); LB 0.96073 | **keep, pending A1/A2** |
| Blend of all own submissions | LB 0.96079 | hedge |

## Appendix: earlier steps (commented out, do not run)

In [ ]:
# Diagnostics:        !python s6e10_v4.py --stage diag
# Ablation:           !python s6e10_v4.py --stage ablate [--og auto]
# NN bench (fold 1):  !python s6e10_v4.py --stage bench --only tabm_reg,tabm_v1,tabm_v2,tabm_v3
# Smoke test:         !python s6e10_v4.py --quick --stack2 --out quick_out
# Nested NN-margin + pseudo-label experiment: !python s6e10_v4.py --stage boost --folds 5 --nn-name tabm_reg --out boost5
# Seed bagging:       !python s6e10_v4.py --folds 5 --nn-seeds 3 --only tabm_reg --out seeds3
# pytabkit members (realmlp_long, tabm_pk) need:  !pip install -q pytabkit
# Re-stack saved runs without training:  --only tabm_reg --import-npz <file>[,<file>...]